# MemAE-XGBoost IDS2 - Kaggle Runner

Dùng cho Kaggle với project ở `/kaggle/working/memAE-XGboost-IDS` và dataset ở `/kaggle/input/datasets/envyiu/cicids2017`.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = '/kaggle/working/memAE-XGboost-IDS'
RAW_DATA_DIR = '/kaggle/input/datasets/envyiu/cicids2017'

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Working directory:', os.getcwd())
print('Raw data dir:', RAW_DATA_DIR)
print('\nProject files:')
for path in sorted(Path(PROJECT_DIR).iterdir()):
    print(' ', path.name)
print('\nDataset sample files:')
for path in sorted(Path(RAW_DATA_DIR).glob('*'))[:20]:
    print(' ', path.name)

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('GPU không khả dụng, bật Accelerator GPU trong Kaggle Notebook settings')

In [ ]:
# Kaggle thường đã có nhiều package, cell này chỉ cài/upgrade phần còn thiếu.
!pip install -q \
    "pandas>=2.0" \
    "numpy>=1.24" \
    "pyarrow>=12.0" \
    "scikit-learn>=1.3,<2.0" \
    "xgboost>=2.0,<4.0" \
    "matplotlib>=3.7" \
    "seaborn>=0.12" \
    "pyyaml>=6.0" \
    "joblib>=1.3" \
    "tqdm>=4.65" \
    "scipy>=1.10"


In [ ]:
# Cấu hình chạy
FAMILIES = 'all'          # test nhanh: 'botnet' hoặc 'dos'
FORCE_RETRAIN = False
CLEAN_DATA = True
PREPROCESS_DEVICE = 'cuda'       # Kaggle GPU: chọn thẳng cuda để fail sớm nếu chưa bật GPU
MEMAE_CONFIG = 'configs/memae_kaggle_t4x2.yaml'
XGBOOST_CONFIG = 'configs/xgboost_kaggle_gpu.yaml'
PREPROCESS_BATCH_ROWS = 524288    # giảm 262144/131072 nếu CUDA OOM
PREPROCESS_FIT_SAMPLE_ROWS = 1000000
PREPROCESS_TMP_DIR = '/kaggle/working/ids2_preprocess_tmp'
MEMAE_EXPORT_BATCH_SIZE = 16384

print('families:', FAMILIES)
print('force_retrain:', FORCE_RETRAIN)
print('clean_data:', CLEAN_DATA)
print('preprocess_device:', PREPROCESS_DEVICE)
print('preprocess_tmp:', PREPROCESS_TMP_DIR)
print('preprocess_fit_rows:', PREPROCESS_FIT_SAMPLE_ROWS)
print('memae_config:', MEMAE_CONFIG)
print('xgboost_config:', XGBOOST_CONFIG)

In [ ]:
import os, subprocess, sys, time

cmd = [
    sys.executable, '-u', 'scripts/run_full_pipeline_all_families.py',
    '--raw-data-dir', RAW_DATA_DIR,
    '--families', FAMILIES,
    '--memae-config', MEMAE_CONFIG,
    '--xgboost-config', XGBOOST_CONFIG,
    '--window-config', 'configs/window_features_zdr5.yaml',
    '--experiment-suffix', 'host_disjoint_zdr5',
    '--variant-suffix', 'targetsel_zdr5',
    '--fusion-suffix', 'scorefusion',
    '--benchmark-mode', 'host_disjoint_window',
    '--split-group-mode', 'host',
    '--fpr-budgets', '0.001,0.005,0.01,0.02,0.05',
    '--max-observed-test-fpr', '0.05',
    '--preprocess-device', PREPROCESS_DEVICE,
    '--preprocess-batch-rows', str(PREPROCESS_BATCH_ROWS),
    '--preprocess-fit-sample-rows', str(PREPROCESS_FIT_SAMPLE_ROWS),
    '--preprocess-tmp-dir', PREPROCESS_TMP_DIR,
    '--memae-export-batch-size', str(MEMAE_EXPORT_BATCH_SIZE),
    '--memae-export-data-parallel',
    '--memae-export-amp',
    '--memae-export-num-workers', '2',
]
if FORCE_RETRAIN:
    cmd.append('--force-retrain')
if CLEAN_DATA:
    cmd.append('--clean-data')

env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONUNBUFFERED'] = '1'

print('Command:', ' '.join(cmd))
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'Exit code: {proc.returncode} | Elapsed: {(time.time() - t0) / 60:.1f} min')

In [ ]:
from pathlib import Path
import glob, os

report_dirs = sorted(glob.glob(os.path.join(PROJECT_DIR, 'reports/run_*')))
if report_dirs:
    latest = report_dirs[-1]
    print('Latest run:', latest)
    for path in sorted(Path(latest).glob('*summary.md')):
        print('\n===', path.name, '===')
        print(path.read_text())
else:
    print('Chưa có report nào')

!df -h /kaggle/working
!du -sh /kaggle/working/ids2_preprocess_tmp 2>/dev/null || true
!du -sh /kaggle/working/memAE-XGboost-IDS/data /kaggle/working/memAE-XGboost-IDS/artifacts /kaggle/working/memAE-XGboost-IDS/reports 2>/dev/null || true